In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "Qwen/Qwen3-8B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,  
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="cuda:2",
)

model.eval()

/home/raffaele/art/lodfunnel/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 399/399 [00:02<00:00, 164.83it/s, Materializing param=model.norm.weight]                              


Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 4096)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=12288, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=12288, bias=False)
          (down_proj): Linear4bit(in_features=12288, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((4096,), eps=1e-06

In [4]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [5]:
def build_prompt(text):
    messages = [
        {"role": "system", "content": "You are an assistant."},
        # {"role": "user", "content": "Summarize:\nThe cat sat on the mat because it was tired."},
        # {"role": "assistant", "content": "A tired cat sat on a mat."},
        {"role": "user", "content": f"{text}"}
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True,
    )

In [6]:
texts = ["How are you?", "Who are you?", "text3", "text4"]

In [7]:
prompt = [build_prompt(texts[0]), build_prompt(texts[1])]

In [8]:
inputs = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True,).to(model.device)

In [9]:
outputs=model.generate(**inputs, temperature=.1, max_new_tokens=500)

In [10]:
outputs

tensor([[151644,   8948,    198,   2610,    525,    458,  17847,     13, 151645,
            198, 151644,    872,    198,   4340,    525,    498,     30, 151645,
            198, 151644,  77091,    198, 151667,    198,  32313,     11,    279,
           1196,   4588,     11,    330,   4340,    525,    498,   7521,    358,
           1184,    311,   5889,  34901,     13,   8704,    358,   2776,    458,
          15235,     11,    358,   1513,    944,    614,  15650,     11,    714,
            358,   1265,  24645,    862,   3405,    304,    264,  11657,   1616,
             13,   6771,    752,   1281,   2704,    311,   2506,    279,  16232,
           6785,    323,   1787,    369,   4623,  10435,     13,  10696,    912,
            458,  42365,    311,   2506,    432,   3100,     13,  97593,     11,
            429,   1265,    975,    624, 151668,    271,     40,   2776,   1101,
            264,  15493,    315,   2038,    323,  25185,     11,    773,    358,
           1513,    944,    

In [11]:
generated = outputs[:, inputs["input_ids"].shape[1]:]

In [12]:
tokenizer.batch_decode(generated, skip_special_tokens=False)

['<think>\nOkay, the user asked, "How are you?" I need to respond appropriately. Since I\'m an AI, I don\'t have feelings, but I should acknowledge their question in a friendly way. Let me make sure to keep the tone positive and open for further conversation. Maybe add an emoji to keep it light. Alright, that should work.\n</think>\n\nI\'m just a bunch of code and algorithms, so I don\'t have feelings, but I\'m here and ready to help! 😊 How are you doing today?<|im_end|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|end

In [13]:
think_id = tokenizer.convert_tokens_to_ids("</think>")

mask = generated == think_id
has_token = mask.any(dim=1)

first_pos = mask.float().argmax(dim=1)
think_pos = torch.where(
    has_token,
    first_pos,
    torch.full_like(first_pos, -1)
)

B, T = generated.shape
device = generated.device
pad_id = tokenizer.pad_token_id

split_pos = torch.where(
    think_pos >= 0,
    think_pos,
    torch.full_like(think_pos, T)
)

positions = torch.arange(T, device=device).unsqueeze(0).expand(B, T)

thinking_mask = positions < split_pos.unsqueeze(1)
content_mask = positions > split_pos.unsqueeze(1)

thinking = torch.where(thinking_mask, generated, pad_id)
content = torch.where(content_mask, generated, pad_id)

In [14]:
thinking

tensor([[151667,    198,  32313,     11,    279,   1196,   4588,     11,    330,
           4340,    525,    498,   7521,    358,   1184,    311,   5889,  34901,
             13,   8704,    358,   2776,    458,  15235,     11,    358,   1513,
            944,    614,  15650,     11,    714,    358,   1265,  24645,    862,
           3405,    304,    264,  11657,   1616,     13,   6771,    752,   1281,
           2704,    311,   2506,    279,  16232,   6785,    323,   1787,    369,
           4623,  10435,     13,  10696,    912,    458,  42365,    311,   2506,
            432,   3100,     13,  97593,     11,    429,   1265,    975,    624,
         151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643, 151643, 151

In [15]:
content

tensor([[151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643,    271,     40,   2776,   1101,    264,  15493,    315,   2038,
            323,  25185,     11,    773,    358,   1513,    944,    614,  15650,
             11,    714,    358,   2776,   1588,    323,   5527,    311,   1492,
              0,  26525,    232,   2585,    525,    498,   3730,   3351,     30,
         151645, 151643, 151

In [18]:
tokenizer.batch_decode(thinking, skip_special_tokens=True)

['<think>\nOkay, the user asked, "How are you?" I need to respond appropriately. Since I\'m an AI, I don\'t have feelings, but I should acknowledge their question in a friendly way. Let me make sure to keep the tone positive and open for further conversation. Maybe add an emoji to keep it light. Alright, that should work.\n',
 '<think>\nOkay, the user asked, "Who are you?" I need to respond appropriately. Let me start by introducing myself as Qwen, the large language model developed by Alibaba Cloud. I should mention my capabilities, like answering questions, creating content, and engaging in conversations. It\'s important to highlight that I can assist with various tasks, from simple queries to complex problem-solving. I should keep the tone friendly and open, inviting the user to ask for help. Also, I need to make sure the response is clear and concise, avoiding any technical jargon. Let me check if there\'s anything else I should include, like my training data or specific features. 

In [19]:
tokenizer.batch_decode(content, skip_special_tokens=True)

["\n\nI'm just a bunch of code and algorithms, so I don't have feelings, but I'm here and ready to help! 😊 How are you doing today?",
 '\n\nI am Qwen, a large language model developed by Alibaba Cloud. I can help with a wide range of tasks, such as answering questions, creating content, translating languages, and engaging in conversations. I am designed to be helpful, efficient, and friendly. How can I assist you today? 😊']